# Support Vector Machines — Lagrange Duality and Optimal Margin Classifier

## Learning Objectives
- Formulate the **primal QP** for the optimal margin classifier and explain why it is hard to solve directly
- Define the **generalised Lagrangian** for problems with inequality constraints
- State the **primal** and **dual** optimisation problems and explain when $d^* = p^*$ (strong duality)
- Derive the **KKT conditions** (stationarity, primal feasibility, dual feasibility, complementarity)
- Apply Lagrange duality to the SVM primal to obtain the **dual QP** in $\\alpha$
- Identify **support vectors** from the KKT dual-complementarity condition
- Recover $w^*$ and $b^*$ from the optimal $\\alpha^*$

## Problem Setup

We have a linearly separable training set

$$\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^{n}, \quad x^{(i)} \in \mathbb{R}^d,\; y^{(i)} \in \{-1, +1\}$$

The SVM seeks the hyperplane $\{x : w^T x + b = 0\}$ that maximises the geometric margin.
From `ml_001_15_2` (Margins notebook), under the canonical form ($\min_i \hat{\gamma}^{(i)} = 1$),
maximising $\gamma = 1/\|w\|$ is equivalent to:

$$\boxed{\min_{w,\,b}\; \tfrac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y^{(i)}(w^T x^{(i)} + b) \geq 1,\; i=1,\ldots,n}$$

This is a **convex QP** with a quadratic objective and $n$ linear inequality constraints.

| Symbol | Meaning |
|---|---|
| $w \in \mathbb{R}^d$ | Normal to the decision hyperplane |
| $b \in \mathbb{R}$ | Bias |
| $\alpha_i \geq 0$ | Lagrange multiplier for the $i$-th constraint |
| $\mathcal{L}$ | Generalised Lagrangian |
| $p^*$ | Value of the primal problem |
| $d^*$ | Value of the dual problem |

## 1. Lagrange Duality

### 1.1 The general constrained optimisation problem

Consider the **primal problem**:

$$\min_w\; f(w) \quad \text{s.t.} \quad g_i(w) \leq 0,\; i=1,\ldots,k \qquad h_j(w) = 0,\; j=1,\ldots,l$$

### 1.2 The generalised Lagrangian

$$\mathcal{L}(w, \alpha, \beta) = f(w) + \sum_{i=1}^{k} \alpha_i g_i(w) + \sum_{j=1}^{l} \beta_j h_j(w)$$

with $\alpha_i \geq 0$.  Define

$$\theta_{\mathcal{P}}(w) = \max_{\alpha \geq 0,\, \beta}\; \mathcal{L}(w,\alpha,\beta)$$

If $w$ **violates** any primal constraint then $\theta_{\mathcal{P}}(w) = +\infty$; otherwise $\theta_{\mathcal{P}}(w) = f(w)$.
Hence the primal problem is equivalent to $\min_w \theta_{\mathcal{P}}(w)$.

### 1.3 The dual problem

$$\theta_{\mathcal{D}}(\alpha, \beta) = \min_w\; \mathcal{L}(w, \alpha, \beta)$$

The **dual optimisation problem** is:

$$\max_{\alpha \geq 0,\, \beta}\; \theta_{\mathcal{D}}(\alpha, \beta)$$

### 1.4 Weak and strong duality

It always holds that $d^* \leq p^*$ (weak duality).  Under the **Slater conditions**:

- $f$ and the $g_i$ are convex
- The $h_j$ are affine (linear + constant)
- There exists a strictly feasible $w$ (i.e., $g_i(w) < 0$ for all $i$)

we have **strong duality** $d^* = p^*$, so the dual can be solved instead of the primal.

The SVM primal satisfies all three conditions (convex QP, affine constraints, feasible by assumption of linear separability).

## 2. KKT Conditions

Under strong duality, the optimal primal $w^*$ and dual $\alpha^*, \beta^*$ must satisfy the
**Karush-Kuhn-Tucker (KKT) conditions**:

| Condition | Equation | Name |
|---|---|---|
| (KKT1) | $\dfrac{\partial}{\partial w_j}\mathcal{L}(w^*, \alpha^*, \beta^*) = 0$ | Stationarity |
| (KKT2) | $\dfrac{\partial}{\partial \beta_j}\mathcal{L}(w^*, \alpha^*, \beta^*) = 0$ | Equality constraint |
| (KKT3) | $\alpha_i^* g_i(w^*) = 0$ | **Dual complementarity** |
| (KKT4) | $g_i(w^*) \leq 0$ | Primal feasibility |
| (KKT5) | $\alpha_i^* \geq 0$ | Dual feasibility |

**KKT3 (dual complementarity)** is the key insight for SVMs:

> For each training example, either $\alpha_i^* = 0$ or $g_i(w^*) = 0$.

Since $g_i(w) = -[y^{(i)}(w^T x^{(i)} + b) - 1]$, having $g_i(w^*) = 0$ means the point
sits **exactly on the margin boundary** — these are the **support vectors**.

Points not on the boundary have $\alpha_i^* = 0$ and do not contribute to $w^*$ at all.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Left: schematic of primal vs dual optimisation ──────────────────────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.axis('off')

boxes = [
    (5, 8.5, 'Primal Problem\n$\\min_w f(w)$ s.t. $g_i(w)\\leq 0$', '#d1e5f0', '#2166ac'),
    (5, 6.0, 'Generalised Lagrangian\n$\\mathcal{L}(w,\\alpha)=f(w)+\\sum\\alpha_i g_i(w)$', '#e8f5e9', '#1a9641'),
    (2.0, 3.5, 'Primal function\n$\\theta_{\\mathcal{P}}(w)=\\max_{\\alpha\\geq 0}\\mathcal{L}$\n$=f(w)$ if feasible, $+\\infty$ else', '#fff3e0', '#e65100'),
    (8.0, 3.5, 'Dual function\n$\\theta_{\\mathcal{D}}(\\alpha)=\\min_w\\mathcal{L}$\n(concave in $\\alpha$)', '#fce4ec', '#c62828'),
    (5, 1.0, 'Strong duality: $d^*=p^*$ when Slater holds\nSolve dual instead of primal!', '#ede7f6', '#4527a0'),
]
for (x, y, txt, fc, ec) in boxes:
    ax.text(x, y, txt, ha='center', va='center', fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=fc, edgecolor=ec, lw=2))

arrows = [(5,7.8,5,6.6), (5,5.4,2.0,4.2), (5,5.4,8.0,4.2), (2.0,2.8,5,1.5), (8.0,2.8,5,1.5)]
for x1,y1,x2,y2 in arrows:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#444', lw=1.8, mutation_scale=16))

ax.set_title('Primal $\\leftrightarrow$ Dual Relationship', fontsize=12, fontweight='bold')

# ── Right: weak vs strong duality on an example ─────────────────────────
ax = axes[1]
# 1-D example: f(w) = (w-2)^2, g(w) = w - 3 <= 0
# Lagrangian: L(w,a) = (w-2)^2 + a*(w-3)
# Dual: theta_D(a) = min_w L = min (w-2)^2 + a*(w-3)
# dL/dw = 2(w-2)+a = 0 => w* = 2 - a/2
# theta_D(a) = (a/2)^2 - a*(2-a/2+3-2) = a^2/4 - a*(1+a/2) 
# Actually: L(w,a) at w*=2-a/2: ((2-a/2)-2)^2 + a*((2-a/2)-3) = a^2/4 + a*(-1-a/2) = a^2/4 - a - a^2/2 = -a^2/4 - a
a_vals = np.linspace(0, 10, 200)
theta_D = -a_vals**2/4 - a_vals  # dual function
# primal solution: w* = 3 (constrained), p* = (3-2)^2 = 1
p_star = 1.0

ax.plot(a_vals, theta_D, 'b-', lw=2.5, label=r'$\theta_{\mathcal{D}}(\alpha)$ (dual function)')
ax.axhline(p_star, color='red', lw=2, ls='--', label=f'$p^* = {p_star}$  (primal value)')
# find d*
d_star = theta_D.max()
a_opt = a_vals[theta_D.argmax()]
ax.scatter([a_opt], [d_star], s=120, color='#1a9641', zorder=5,
           label=f'$d^* = {d_star:.2f}$ at $\\alpha^*={a_opt:.1f}$')
ax.fill_between(a_vals, theta_D, p_star, alpha=0.12, color='orange', label='duality gap')
ax.set_xlabel(r'$\alpha$ (Lagrange multiplier)', fontsize=11)
ax.set_ylabel('Objective value', fontsize=11)
ax.set_title('Weak Duality: $d^* \\leq p^*$\n(gap exists when Slater fails)', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.05, 0.08, 'For SVM: Slater holds\n=> strong duality $d^*=p^*$',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', fc='#ede7f6', alpha=0.9))

fig.suptitle('Lagrange Duality: Primal, Dual, and Strong Duality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Applying Lagrange Duality to the SVM Primal

### 3.1 Rewrite constraints in standard form

The SVM primal:

$$\min_{w,b}\; \tfrac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y^{(i)}(w^T x^{(i)} + b) \geq 1, \; i=1,\ldots,n$$

Rewrite constraints as $g_i(w) \leq 0$:

$$g_i(w, b) = -y^{(i)}(w^T x^{(i)} + b) + 1 \leq 0$$

### 3.2 The SVM Lagrangian

$$\mathcal{L}(w, b, \alpha) = \frac{1}{2}\|w\|^2 - \sum_{i=1}^{n} \alpha_i\bigl[y^{(i)}(w^T x^{(i)} + b) - 1\bigr]$$

with $\alpha_i \geq 0$ (no $\beta$ terms since there are no equality constraints).

### 3.3 Minimise over $w$ and $b$ (KKT stationarity)

**With respect to $w$:**

$$\nabla_w \mathcal{L} = w - \sum_{i=1}^n \alpha_i y^{(i)} x^{(i)} = 0 \quad \Longrightarrow \quad \boxed{w = \sum_{i=1}^n \alpha_i y^{(i)} x^{(i)}}$$

**With respect to $b$:**

$$\frac{\partial \mathcal{L}}{\partial b} = -\sum_{i=1}^n \alpha_i y^{(i)} = 0 \quad \Longrightarrow \quad \boxed{\sum_{i=1}^n \alpha_i y^{(i)} = 0}$$

### 3.4 Substitute into the Lagrangian

Substituting $w = \sum_i \alpha_i y^{(i)} x^{(i)}$ and using $\sum_i \alpha_i y^{(i)} = 0$:

$$\mathcal{L} = \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} y^{(i)} y^{(j)} \alpha_i \alpha_j (x^{(i)})^T x^{(j)}$$

This is $\theta_{\mathcal{D}}(\alpha)$ — the **dual objective** — which depends only on $\alpha$.

## 4. The SVM Dual QP

After the substitution, the dual optimisation problem is:

$$\boxed{\max_{\alpha}\; W(\alpha) = \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i=1}^n\sum_{j=1}^n y^{(i)} y^{(j)} \alpha_i \alpha_j \langle x^{(i)}, x^{(j)} \rangle}$$

$$\text{s.t.} \quad \alpha_i \geq 0,\; i = 1,\ldots,n \qquad \sum_{i=1}^n \alpha_i y^{(i)} = 0$$

### Why the dual is preferable to the primal

| Property | Primal QP | Dual QP |
|---|---|---|
| Variables | $d + 1$ ($w$ and $b$) | $n$ ($\alpha_i$'s) |
| Constraints | $n$ inequalities | $n$ non-negativity + 1 equality |
| Depends on data via | $x^{(i)}$ directly | inner products $\langle x^{(i)}, x^{(j)} \rangle$ only |
| Enables kernel trick? | No | **Yes** — replace $\langle x^{(i)}, x^{(j)} \rangle$ with $K(x^{(i)}, x^{(j)})$ |

### Kernelised dual

Simply replace each inner product with a kernel evaluation:

$$\max_{\alpha}\; \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i,j} y^{(i)} y^{(j)} \alpha_i \alpha_j K(x^{(i)}, x^{(j)})$$

The feature map $\phi$ is **never computed explicitly** — only the $n \times n$ kernel matrix $K$ is needed.

## 5. Recovering $w^*$, $b^*$, and Making Predictions

### Recovering $w^*$

From the stationarity condition:

$$w^* = \sum_{i=1}^n \alpha_i^* y^{(i)} x^{(i)} = \sum_{i \in \mathcal{SV}} \alpha_i^* y^{(i)} x^{(i)}$$

The second equality holds because $\alpha_i^* = 0$ for all non-support vectors (KKT complementarity).
Only the **support vectors** contribute to $w^*$.

### Recovering $b^*$

Any support vector $x^{(s)}$ satisfies $y^{(s)}(w^{*T} x^{(s)} + b^*) = 1$, so:

$$b^* = y^{(s)} - w^{*T} x^{(s)}$$

For numerical stability, average over all support vectors:

$$b^* = -\frac{\max_{i:y^{(i)}=-1} w^{*T}x^{(i)} + \min_{i:y^{(i)}=1} w^{*T}x^{(i)}}{2}$$

### Making predictions

**Primal form** (when $d$ is small):

$$h(x) = \operatorname{sign}(w^{*T}x + b^*)$$

**Dual / kernelised form** (general):

$$h(x) = \operatorname{sign}\!\left(\sum_{i \in \mathcal{SV}} \alpha_i^* y^{(i)} K(x^{(i)}, x) + b^*\right)$$

Prediction cost is $O(|\mathcal{SV}| \cdot d)$ — only the (often small) support set matters at test time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

rng = np.random.default_rng(42)
pos = rng.normal(loc=[3.5, 3.5], scale=0.5, size=(14, 2))
neg = rng.normal(loc=[1.2, 1.2], scale=0.5, size=(14, 2))
X = np.vstack([pos, neg])
y = np.hstack([np.ones(14), -np.ones(14)])

clf = SVC(kernel='linear', C=1e6)
clf.fit(X, y)
w = clf.coef_[0]
b = clf.intercept_[0]
wn = w / np.linalg.norm(w)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Left: decision boundary with support vectors highlighted ─────────────
ax = axes[0]
xr = np.linspace(-0.5, 5.5, 300)
yr_db  = -(w[0]*xr + b    ) / w[1]
yr_pos = -(w[0]*xr + b - 1) / w[1]
yr_neg = -(w[0]*xr + b + 1) / w[1]
ax.fill_between(xr, yr_neg, yr_pos, alpha=0.12, color='#4dac26')
ax.plot(xr, yr_db,  'k-',  lw=2.2)
ax.plot(xr, yr_pos, 'b--', lw=1.8)
ax.plot(xr, yr_neg, 'r--', lw=1.8)
ax.scatter(pos[:,0], pos[:,1], marker='x', s=80, c='#2166ac', linewidths=2, label='$y=+1$')
ax.scatter(neg[:,0], neg[:,1], marker='o', s=65, facecolors='none',
           edgecolors='#d6604d', linewidths=2, label='$y=-1$')
sv = clf.support_vectors_
ax.scatter(sv[:,0], sv[:,1], s=280, facecolors='none',
           edgecolors='#ff7f00', linewidths=2.5, zorder=5, label='support vectors ($\\alpha_i>0$)')
ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('KKT Complementarity:\nOnly Support Vectors Have $\\alpha_i > 0$', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.03, 0.03,
        f'{len(sv)} support vectors out of {len(X)} training points\n'
        f'All others: $\\alpha_i^*=0$, do not affect $w^*$',
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# ── Right: w reconstruction from support vectors ─────────────────────────
ax = axes[1]
# simulate alpha values (proportional to distance from boundary for illustration)
sv_labels = np.array([1.0 if (w@sv_i+b) > 0 else -1.0 for sv_i in sv])
# functional margins of support vectors should be ~1
func_margins = sv_labels * (sv @ w + b)
ax.bar(range(len(sv)), func_margins, color=['#2166ac' if l>0 else '#d6604d' for l in sv_labels],
       alpha=0.8, label='functional margin (should be ~1.0)')
ax.axhline(1.0, color='k', lw=2, ls='--', label='canonical margin = 1')
ax.set_xlabel('Support vector index', fontsize=10)
ax.set_ylabel('Functional margin $y^{(i)}(w^T x^{(i)}+b)$', fontsize=9)
ax.set_title('Support Vectors Sit on the Margin Boundary\n$y^{(i)}(w^Tx^{(i)}+b) = 1$ (KKT Eq. 3)', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.03, 0.03,
        'KKT complementarity: $g_i(w^*)=0$\nfor all support vectors',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('SVM Dual Solution: Support Vectors and KKT Complementarity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Numerical Verification

The code below verifies the key relationships:
1. $w^* = \\sum_i \\alpha_i^* y^{(i)} x^{(i)}$ (primal recovered from dual)
2. Support vectors satisfy $y^{(i)}(w^{*T}x^{(i)}+b^*) \\approx 1$ (KKT complementarity)
3. Non-support vectors have $\\alpha_i^* \\approx 0$
4. Strong duality: primal and dual objectives match at the optimum

In [ ]:
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(7)
pos = rng.normal(loc=[3.0, 3.0], scale=0.5, size=(6, 2))
neg = rng.normal(loc=[0.8, 0.8], scale=0.5, size=(6, 2))
X = np.vstack([pos, neg])
y = np.hstack([np.ones(6), -np.ones(6)])
n = len(y)

# ── Solve dual QP via scipy (negated for minimisation) ────────────────────
def neg_dual(alpha):
    W = np.sum(alpha) - 0.5 * np.sum(
        (alpha[:, None] * alpha[None, :]) *
        (y[:, None] * y[None, :]) *
        (X @ X.T)
    )
    return -W

def neg_dual_grad(alpha):
    grad = np.ones(n) - (alpha * y) @ (np.outer(y, y) * (X @ X.T))
    return -grad

constraints = [{'type': 'eq', 'fun': lambda a: a @ y, 'jac': lambda a: y}]
bounds = [(0, None)] * n
a0 = np.ones(n) * 0.1
res = minimize(neg_dual, a0, jac=neg_dual_grad,
               method='SLSQP', bounds=bounds, constraints=constraints)
alpha_star = np.maximum(res.x, 0)

# ── Recover w* and b* from alpha* ─────────────────────────────────────────
w_dual = (alpha_star * y) @ X
sv_mask = alpha_star > 1e-5
b_star = np.mean(y[sv_mask] - X[sv_mask] @ w_dual)

# ── Compare with sklearn ──────────────────────────────────────────────────
from sklearn.svm import SVC
clf = SVC(kernel='linear', C=1e6)
clf.fit(X, y)
w_primal = clf.coef_[0]
b_primal = clf.intercept_[0]

print('=== Dual QP Solution ===')
print(f'  alpha values (non-zero = support vectors):')
for i, (a, yi) in enumerate(zip(alpha_star, y)):
    sv = '*SV*' if a > 1e-5 else ''
    print(f'    i={i:2d}  y={int(yi):+d}  alpha={a:.4f}  {sv}')

print(f'\n  w* from dual  = {w_dual.round(4)}')
print(f'  w* from primal= {w_primal.round(4)}')
print(f'  b* from dual  = {b_star:.4f}')
print(f'  b* from primal= {b_primal:.4f}')

print('\n=== KKT Complementarity Check ===')
margins = y * (X @ w_dual + b_star)
print(f'  Functional margins at support vectors (should be ~1):')
for i in np.where(sv_mask)[0]:
    print(f'    i={i:2d}  margin={margins[i]:.4f}')

print('\n=== Strong Duality Check ===')
primal_obj = 0.5 * np.dot(w_dual, w_dual)
dual_obj = -neg_dual(alpha_star)
print(f'  Primal objective (1/2 ||w||^2) = {primal_obj:.6f}')
print(f'  Dual objective   W(alpha*)     = {dual_obj:.6f}')
print(f'  Duality gap = {abs(primal_obj - dual_obj):.2e}  (should be ~0)')

## 7. Derivation Pathway

```
Primal QP: min 1/2||w||^2  s.t. y^(i)(w^T x^(i)+b) >= 1
   |
   | introduce alpha_i >= 0, form Lagrangian L(w,b,alpha)
   v
KKT stationarity: dL/dw=0 => w = sum alpha_i y^(i) x^(i)
                  dL/db=0 => sum alpha_i y^(i) = 0
   |
   | substitute back into L
   v
Dual QP: max W(alpha) = sum alpha_i - 1/2 sum alpha_i alpha_j y^(i) y^(j) <x^(i),x^(j)>
         s.t. alpha_i >= 0, sum alpha_i y^(i) = 0
   |
   | solve for alpha* (SMO or QP solver)
   v
KKT complementarity: alpha_i [y^(i)(w^T x^(i)+b) - 1] = 0
   => alpha_i > 0 only for support vectors (on margin boundary)
   |
   | recover w* = sum_{SV} alpha_i y^(i) x^(i)
   | recover b* from any support vector
   v
Predict: h(x) = sign(w^T x + b*)
         or kernelised: sign(sum_{SV} alpha_i y^(i) K(x^(i), x) + b*)
```

## Summary

| Concept | Formula / Condition | Role |
|---|---|---|
| SVM Primal QP | $\min \frac{1}{2}\|w\|^2$ s.t. $y^{(i)}(w^Tx^{(i)}+b)\geq 1$ | Original formulation |
| Lagrangian | $\mathcal{L} = \frac{1}{2}\|w\|^2 - \sum \alpha_i[y^{(i)}(w^Tx^{(i)}+b)-1]$ | Bridge primal ↔ dual |
| KKT stationarity | $w = \sum \alpha_i y^{(i)} x^{(i)}$, $\sum \alpha_i y^{(i)}=0$ | Eliminate $w$, $b$ |
| Dual QP | $\max W(\alpha)$, s.t. $\alpha_i\geq 0$, $\sum\alpha_i y^{(i)}=0$ | Solved in practice |
| Strong duality | $d^* = p^*$ (Slater holds for SVM) | Dual solution = primal solution |
| KKT complementarity | $\alpha_i[y^{(i)}(w^Tx^{(i)}+b)-1]=0$ | Identifies support vectors |
| Support vectors | Points with $\alpha_i^* > 0$ | Determine $w^*$ entirely |
| Prediction | $\operatorname{sign}(\sum_{\mathcal{SV}} \alpha_i^* y^{(i)} K(x^{(i)},x) + b^*)$ | Kernelisable |

**Key insight:** strong duality lets us solve the dual QP instead of the primal. The dual depends on inputs
only through inner products, enabling the kernel trick. KKT complementarity reveals that only the
support vectors (a small subset) matter — making SVM predictions fast and memory-efficient.